<a href="https://colab.research.google.com/github/finneKIM/stem-remix-assistant/blob/main/notebooks/02a_regenerate_musicongen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02a — EXP-002: MusiConGen 재생성 파이프라인

Demucs로 분리한 원곡 스템 중 하나(`target_stem`)를 MusiConGen으로 재생성하고, 다시 Demucs로 분리한 뒤 madmom 기반 Alignment Engine으로 원곡과 타이밍을 맞추는 파이프라인.

설정값은 `experiments/exp_002_musicongen/config.yaml` 참고. 파이프라인 전체 구조는 Notion "02번 노트북 — 재생성/재조합 로직 설계" 페이지.

**실행 환경**: Google Colab, GPU 런타임(T4) 필수. MusiConGen 추론에 12GB+ VRAM 권장, T4는 16GB라 충족.

**실행의 목적**: `duration_sec`를 최소(10s)/중간(15s)/최대(20s) 세 구간으로 스윕, `seed`는 42로 전 구간 고정해서 최적 duration_sec 탐색 (2026-08-28 TODO).

## 0. GPU 확인

In [4]:
!nvidia-smi|

/bin/bash: -c: line 2: syntax error: unexpected end of file


In [5]:
!ls -la /content/

total 24
drwxr-xr-x 1 root root 4096 Sep 14 07:17 .
drwxr-xr-x 1 root root 4096 Sep 14 07:11 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 5 root root 4096 Sep 14 07:17 MusiConGen
-rw-r--r-- 1 root root    0 Sep 14 07:17 requirements_no_xformers.txt
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data
drwxr-xr-x 6 root root 4096 Sep 14 07:16 stem-remix-assistant


## 1. 저장소 클론 + MusiConGen 설치

- `stem-remix-assistant`: 원곡/스템 샘플(`docs/samples/`)과 `config.yaml` 확보용
- `MusiConGen`: 공식 저장소 (https://github.com/YatingMusic/MusiConGen)

In [6]:
!git clone https://github.com/finneKIM/stem-remix-assistant.git
!git clone https://github.com/YatingMusic/MusiConGen.git

%cd MusiConGen
!pip install -r requirements.txt -q
!conda install -y 'ffmpeg<5' -c conda-forge 2>/dev/null || apt-get -y install ffmpeg -q


fatal: destination path 'stem-remix-assistant' already exists and is not an empty directory.
fatal: destination path 'MusiConGen' already exists and is not an empty directory.
/content/MusiConGen
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [7]:
!python --version

Python 3.13.15


#### 실험 - requirements.txt에서 xforemrs 줄 제외하고 설치
- requirements.txt를 그대로 쓰지 않고 xformers 줄만 걸러서 설치
- MusiConGen/audiocraft가 xFormers 없이도 정상 임포트되는지 확인
- 실제 생성 확인으로 xFormers 없이도 파이프라인이 도는지 검증

In [8]:
# reuiqrements.txt에서 xFormers 줄만 제외하고 설치
!grep -v -i "xforemrs" requirements.txt > requirements_no_xformers.txt
!cat requirements_no_xformers.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
torch==2.0.0
torchaudio==2.0.0
tqdm
transformers==4.31.0  # need Encodec there.
xformers==0.0.22
demucs
librosa
soundfile
torchmetrics
encodec
protobuf
torchvision==0.16.0
torchtext==0.16.0
pesq
pystoi


In [9]:
# torch 버전과 python 3.13 버전 호환여부 확인
!pip index versions torch 2>&1 | head -5

torch (2.14.0)
Available versions: 2.14.0, 2.13.0, 2.12.1, 2.12.0, 2.11.0, 2.10.0, 2.9.1, 2.9.0, 2.8.0, 2.7.1, 2.7.0, 2.6.0, 2.5.1, 2.5.0
  INSTALLED: 2.11.0+cu128
  LATEST:    2.14.0


In [10]:
import torch, torchaudio, torchvision, torchtext
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("torchvision:", torchvision.__version__)
print("torchtext:", torchtext.__version__)

ModuleNotFoundError: No module named 'torchtext'

In [11]:
# torch, torchaudio, torchvision, torchtext를 requirements.txt에서 제외
# colab 환경의 torch 2.11.0 환경 그대로 나머지 패키지 설치
# import audiocraft가 실제로 torchtext를 요구하는지 에러로 직접 확인

!grep -v -iE "^(xformers|torch|torchaudio|torchvision|torchtext)(==|>=|<)?" requirements.txt > requirements_filtered.txt
!cat requirements_filtered.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [9]:
# numpy 버전 확인 및 그레이드 충돌 or python3.13 wheel 부재 문제 사전확인
# 설치 충동 에러 발생 사전 확인
!pip install -r requirements_filtered.txt

  Using cached av-11.0.0.tar.gz (3.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [12]:
!cat requirements_filtered.txt

av==11.0.0
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [13]:
# av 버전 a안 실행
!sed -i 's/^av==11.0.0/av/' requirements_filtered.txt
!cat requirements_filtered.txt

av
einops
flashy==0.0.1
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [12]:
!pip install -r requirements_filtered.txt -v 2>&1 | tail -100

    Found link https://files.pythonhosted.org/packages/b7/b9/c538f279a4e237a006a2c98387d081e9eb060d203d8ed34467cc0f0b9b53/packaging-26.0-py3-none-any.whl (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.0
    Found link https://files.pythonhosted.org/packages/65/ee/299d360cdc32edc7d2cf530f3accf79c4fca01e96ffc950d8a52213bd8e4/packaging-26.0.tar.gz (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.0
    Found link https://files.pythonhosted.org/packages/7a/c2/920ef838e2f0028c8262f16101ec09ebd5969864e5a64c4c05fad0617c56/packaging-26.1-py3-none-any.whl (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.1
    Found link https://files.pythonhosted.org/packages/df/de/0d2b39fb4af88a0258f3bac87dfcbb48e73fbdea4a2ed0e2213f9a4c2f9a/packaging-26.1.tar.gz (from https://pypi.org/simple/packaging/) (requires-python:>=3.8), version: 26.1
    Found link https://files.pythonhosted.org/packages/df/b2/87e62e8c3e2f4b32e5f

In [14]:
# restarting session해서 requirements_filtered.txt가 사라지지 않게 하는 방법
%cd /content/MusiConGen
!grep -v -iE "^(xformers|torch|torchaudio|torchvision|torchtext)(==|>=|<)?" requirements.txt > /content/requirements_filtered.txt
!sed -i 's/^av==11.0.0/av/' /content/requirements_filtered.txt
!sed -i 's/^flashy==0.0.1/flashy==0.0.2/' /content/requirements_filtered.txt
!cat /content/requirements_filtered.txt

/content/MusiConGen
av
einops
flashy==0.0.2
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [15]:
!pip install -r /content/requirements_filtered.txt -v 2>&1 | tail -100

    Created temporary directory: /tmp/pip-metadata-4t5qv_5l
  Created temporary directory: /tmp/pip-unpack-abdlqch3
  Looking up "https://files.pythonhosted.org/packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl" in the cache
  No cache entry available
  No cache entry available
  https://files.pythonhosted.org:443 "GET /packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl HTTP/1.1" 200 952017
  Ignoring unknown cache-control directive: immutable
  Updating cache with response from "https://files.pythonhosted.org/packages/18/ad/ec41343a49a0371ea40daf37b1ba2c11333cdd121cb378161635d14b9750/setuptools-59.2.0-py3-none-any.whl"
  etag object cached for 1209600 seconds
  Caching due to etag
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.0/952.0 kB 20.3 MB/s eta 0:00:00
  Looking up "https://files.pythonhosted.org/packages/04/80/cad93b40262f5d09f6de82adbee452fd43cdff60830b5

In [1]:
# python 호환문제로 miniconda로 실행 환경 변경
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /content/miniconda3
!/content/miniconda3/bin/conda --version

PREFIX=/content/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3
conda 26.7.1


In [2]:
# miniconda3의 격리된 환경 설정
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

For information on safely removing channels from your conda configuration,
please see the official documentation:

    https://www.anaconda.com/docs/tools/working-with-conda/channels



In [3]:
 # 저장소 약관 동의
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [4]:
# 다시 격리된 환경 설정 및 실행
!/content/miniconda3/bin/conda create -n stemremix python=3.11 -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: | done


==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.7.2

Please update conda by running

    $ conda self update



## Package Plan ##

  environment location: /content/miniconda3/envs/stemremix

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libexpat-2.8.4             |       h7354ed3_0         128 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    openssl-3.5.8              |       h1b28b03_0         5.5 MB
    packaging-26.3             |  py311h06a4308_0         380 KB
    pip-26.2.1                 |     pyhc872135_0         1.

In [5]:
# miniconda 설치 -> gpu, pythohn 버전 확인
!/content/miniconda3/envs/stemremix/bin/python --version
!/content/miniconda3/envs/stemremix/bin/python -c "import sys; print(sys.executable)"
!nvidia-smi

Python 3.11.16
/content/miniconda3/envs/stemremix/bin/python
Mon Sep 14 07:20:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |           

In [6]:
# stemremix 환경 안에 cuda 13.0에 맞는 torch 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 23.1 MB/s  0:00:16
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 129.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 412.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 147.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 40.9 MB/s  0:00:11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 58.9 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 170.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 175.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 181.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 175.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 158

In [7]:
#gpu 인식여부 확인
!/content/miniconda3/envs/stemremix/bin/python -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'gpu 없음')"

/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
2.5.1+cu121
True
Tesla T4


In [8]:
# MusiConGen 파이프라인에 numpy 필수 -> 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 53.0 MB/s  0:00:00


In [9]:
# numpy 설치 확인
!/content/miniconda3/envs/stemremix/bin/python -c "import numpy; print(numpy.__version__)"

2.4.6


### requirements_filtered.txt 재설치

In [10]:
# 파일 확인
!cat /content/requirements_filtered.txt

av
einops
flashy==0.0.2
hydra-core==1.1
hydra_colorlog
julius
num2words
numpy==1.24.4
sentencepiece
spacy==3.6.1
tqdm
transformers==4.31.0  # need Encodec there.
demucs
librosa
soundfile
encodec
protobuf
pesq
pystoi


In [11]:
# 설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install -r /content/requirements_filtered.txt

  Using cached av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached flashy-0.0.2.tar.gz (72 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached hydra_core-1.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached hydra_colorlog-1.2.0-py3-none-any.whl.metadata (949 bytes)
  Using cached julius-0.2.8-py3-none-any.whl.metadata (7.6 kB)
  Using cached num2words-0.5.14-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ...

In [12]:
# 설치된 버전 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip list

Package                  Version
------------------------ -----------
annotated-types          0.8.0
antlr4-python3-runtime   4.8
audioread                3.1.0
av                       18.1.0
blis                     0.7.11
catalogue                2.0.10
certifi                  2026.7.22
cffi                     2.1.1
charset-normalizer       3.5.1
click                    8.5.0
cloudpickle              3.1.2
colorlog                 6.12.0
confection               0.1.5
cymem                    2.0.13
decorator                5.3.1
demucs                   4.1.0
docopt                   0.6.2
dora_search              0.1.13
einops                   0.8.2
encodec                  0.1.1
filelock                 3.32.3
flashy                   0.0.2
fsspec                   2026.7.0
hf-xet                   1.6.0
huggingface_hub          0.36.2
hydra-colorlog           1.2.0
hydra-core               1.1.0
idna                     3.19
Jinja2                   3.1.6
joblib             

In [13]:
# 버전 파일로 저장
!/content/miniconda3/envs/stemremix/bin/python -m pip freeze > /content/stemremix_installed_versions.txt
!cat /content/stemremix_installed_versions.txt

annotated-types==0.8.0
antlr4-python3-runtime==4.8
audioread==3.1.0
av==18.1.0
blis==0.7.11
catalogue==2.0.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
colorlog==6.12.0
confection==0.1.5
cymem==2.0.13
decorator==5.3.1
demucs==4.1.0
docopt==0.6.2
dora_search==0.1.13
einops==0.8.2
encodec==0.1.1
filelock==3.32.3
flashy==0.0.2
fsspec==2026.7.0
hf-xet==1.6.0
huggingface_hub==0.36.2
hydra-colorlog==1.2.0
hydra-core==1.1.0
idna==3.19
Jinja2==3.1.6
joblib==1.6.0
julius==0.2.8
lameenc==1.8.4
langcodes==3.5.1
lazy-loader==0.5
librosa==0.11.0
llvmlite==0.49.0
MarkupSafe==3.0.3
mpmath==1.3.0
msgpack==1.2.2
murmurhash==1.0.15
narwhals==2.26.0
networkx==3.6.1
num2words==0.5.14
numba==0.67.0
numpy==1.24.4
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107
n

In [14]:
# drvie mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
#MusiConGen 폴더 원본 확인
%cd /content/MusiConGen
!git status

/content/MusiConGen
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	requirements_filtered.txt
	requirements_no_xformers.txt

nothing added to commit but untracked files present (use "git add" to track)


In [16]:
!ls -la /content/drive/MyDrive/stem-remix-assistant/
!ls -la /content/requirements_filtered.txt /content/stemremix_installed_versions.txt

total 13980
drwx------ 3 root root     4096 Aug 14 04:06  .
drwx------ 5 root root     4096 Sep 14 07:24  ..
-rw------- 1 root root 14300011 Aug 28 08:52 'Copy of 01_setup_and_test.ipynb'
drwx------ 2 root root     4096 Aug 14 04:06  outputs
-rw------- 1 root root      216 Sep  2 08:11  requirements_filtered.txt
-rw------- 1 root root     1878 Sep  2 08:11  stemremix_installed_versions.txt
-rw-r--r-- 1 root root  216 Sep 14 07:18 /content/requirements_filtered.txt
-rw-r--r-- 1 root root 1879 Sep 14 07:24 /content/stemremix_installed_versions.txt


In [17]:
# /content/의 하위 .txt를 복사
!cp -v /content/requirements_filtered.txt /content/drive/MyDrive/stem-remix-assistant/
!cp -v /content/stemremix_installed_versions.txt /content/drive/MyDrive/stem-remix-assistant/
!ls -la /content/drive/MyDrive/stem-remix-assistant/

'/content/requirements_filtered.txt' -> '/content/drive/MyDrive/stem-remix-assistant/requirements_filtered.txt'
'/content/stemremix_installed_versions.txt' -> '/content/drive/MyDrive/stem-remix-assistant/stemremix_installed_versions.txt'
total 13980
drwx------  3 root root     4096 Sep 14 07:25  .
drwx------ 11 root root     4096 Sep 14 07:24  ..
-rw-------  1 root root 14300011 Aug 28 08:52 'Copy of 01_setup_and_test.ipynb'
drwx------  2 root root     4096 Aug 14 04:06  outputs
-rw-------  1 root root      216 Sep 14 07:25  requirements_filtered.txt
-rw-------  1 root root     1879 Sep 14 07:25  stemremix_installed_versions.txt


### audiocraft import

In [18]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio.py", line 25, in <module>\n    from .audio_utils import f32_pcm, i16_pcm, normalize_audio\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio_utils.py", line 16, in <module>\n    from .chords import Chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/chords.py", line 33, in <module>\n    import pandas as pd\nModuleNotFoundError: No module named \'pandas\'\n'


In [19]:
!ls /content/MusiConGen/
print("===============")
!ls /content/MusiConGen/audiocraft/
print("===============")
!ls /content/MusiConGen/audiocraft/audiocraft/

5_genre_songs_list.json  README.md
audiocraft		 requirements_filtered.txt
LICENSE			 requirements_no_xformers.txt
preproc			 requirements.txt
audiocraft  config  dataset  egs  export_weight.py  generate_chord_beat.py
adversarial	grids	     metrics  optim	   quantization  utils
data		__init__.py  models   __pycache__  solvers
environment.py	losses	     modules  py.typed	   train.py


In [20]:
# pandas 의존성 확인 -> requirements_filterd.txt 기존에는 없음
!head -50 /content/MusiConGen/audiocraft/audiocraft/data/chords.py

# encoding: utf-8
"""
This module contains chord evaluation functionality.

It provides the evaluation measures used for the MIREX ACE task, and
tries to follow [1]_ and [2]_ as closely as possible.

Notes
-----
This implementation tries to follow the references and their implementation
(e.g., https://github.com/jpauwels/MusOOEvaluator for [2]_). However, there
are some known (and possibly some unknown) differences. If you find one not
listed in the following, please file an issue:

 - Detected chord segments are adjusted to fit the length of the annotations.
   In particular, this means that, if necessary, filler segments of 'no chord'
   are added at beginnings and ends. This can result in different segmentation
   scores compared to the original implementation.

References
----------
.. [1] Christopher Harte, "Towards Automatic Extraction of Harmony Information
       from Music Signals." Dissertation,
       Department for Electronic Engineering, Queen Mary University of London,
  

In [21]:
# 코드가 특정 구버전 api에 의존하는지 확인
!grep -n "pd\." /content/MusiConGen/audiocraft/audiocraft/data/chords.py

464:        df = pd.DataFrame(data=entry[['root', 'is_major']])


In [22]:
# pandas latest ver. installation
!/content/miniconda3/envs/stemremix/bin/python -m pip install pandas

  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.0 MB/s  0:00:00
Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]


In [23]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/audio_dataset.py", line 33, in <module>\n    import dora\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/dora/__init__.py", line 68, in <module>\n    import hydra\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/hydra/__init__.py", line 5, in <module>\n    from hydra import utils\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/hydra/utils.py", line 8, in <module>\n    import hydra._internal.instantiate._instantiate2\n  File "/conte

In [24]:
# 재검증 -> hydra-core 호환성 문제
!/content/miniconda3/envs/stemremix/bin/python -m pip install "hydra-core==1.3.2" --force-reinstall

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (806 kB)
Using cached packaging-26.3-py3-none-any.whl (129 kB)
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=a13d9478bee8b91edac0966752dc1bc7012a363138e093211ca69121761c1a4b
  Stored in directory: /root/.cache/pip/wheels/1a/97/32/461f837398029ad76911109f07047fde1d7b661a147c7c56d1
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.8
    Uninstalling antlr4-python3-runtime-4.8:
      Successfully 

In [25]:
# hydra-core 다운그레이드 이후 재검증
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/__init__.py", line 22, in <module>\n    from .transformer import StreamingTransformer\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/transformer.py", line 23, in <module>\n    from xformers import ops\nModuleNotFoundError: No module named \'xformers\'\n'


In [26]:
# transformers가 사용하는 xformers 확인 -> 필수/선택 의존성 여부 확인
!sed -n '1,40p' /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

"""
Transformer model, with streaming support, xformer attention support
and easy causal attention with a potentially finite receptive field.

See `StreamingTransformer` for more information.

Unlike regular PyTorch Transformer, we make the hard choice that batches are first.
"""

import typing as tp

from einops import rearrange
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from xformers import ops

from .rope import RotaryEmbedding
from .streaming import StreamingModule

_efficient_attention_backend: str = 'torch'


def set_efficient_attention_backend(backend: str = 'torch'):
    # Using torch by default, it seems a bit faster on older P100 GPUs (~20% faster).
    global _efficient_attention_backen

In [27]:
# ops 모듈 호출 위치 확인
!grep -n "ops\." /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

373:                    q, k, v = ops.unbind(packed, dim=2)
407:                    x = ops.memory_efficient_attention(q, k, v, attn_mask, p=p)


In [28]:
!sed -n '355,410p' /content/MusiConGen/audiocraft/audiocraft/modules/transformer.py

                k = nn.functional.linear(key, self.in_proj_weight[dim: 2 * dim], bias_k)
                v = nn.functional.linear(value, self.in_proj_weight[2 * dim:], bias_v)
                if self.qk_layer_norm is True:
                    q = self.q_layer_norm(q)
                    k = self.k_layer_norm(k)
                q, k, v = [rearrange(x, f"b t (h d) -> {layout}", h=self.num_heads) for x in [q, k, v]]
            else:
                if not _is_profiled():
                    # profiling breaks that propertysomehow.
                    assert query is key, "specialized implementation"
                    assert value is key, "specialized implementation"
                projected = nn.functional.linear(query, self.in_proj_weight, self.in_proj_bias)
                if self.kv_repeat == 1:
                    if time_dim == 2:
                        bound_layout = "b h p t d"
                    else:
                        bound_layout = "b t p h d"
                    pac

In [29]:
# xformers installation
!/content/miniconda3/envs/stemremix/bin/python -m pip install xformers --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 31.7 MB/s  0:00:00


In [30]:
# audiocraft 재검증
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py", line 6, in <module>\n    import pretty_midi\nModuleNotFoundError: No module named \'pretty_midi\'\n'


In [31]:
# import 구문 전체 grep해서 버전 확인
!grep -rhE "^(import |from )" /content/MusiConGen/audiocraft/audiocraft/ | sort -u

grep: /content/MusiConGen/audiocraft/audiocraft/models/__pycache__/builders.cpython-311.pyc: binary file matches
grep: /content/MusiConGen/audiocraft/audiocraft/solvers/__pycache__/builders.cpython-311.pyc: binary file matches
from abc import ABC, abstractmethod
from audiocraft.data.audio import audio_write
from audiocraft.data.audio_utils import convert_audio
from audiocraft import __version__
from audiocraft.modules.transformer import StreamingTransformer, create_sin_embedding
from .audio_dataset import AudioDataset, AudioMeta
from .audiogen import AudioGen
from .audiogen import AudioGenSolver
from .audio import audio_read, audio_info
from .audio_utils import convert_audio
from .audio_utils import f32_pcm, i16_pcm, normalize_audio
from .balancer import Balancer
from .._base_explorers import BaseExplorer
from .base import BaseQuantizer, DummyQuantizer, QuantizedResult
from .base import BaseQuantizer, QuantizedResult
from .base import MultiDiscriminator, MultiDiscriminatorOutputType
fr

In [32]:
# 현재 stemremix 환경에 설치된 패키지 목록 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip list --format=freeze

annotated-types==0.8.0
antlr4-python3-runtime==4.9.3
audioread==3.1.0
av==18.1.0
blis==0.7.11
catalogue==2.0.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
click==8.5.0
cloudpickle==3.1.2
colorlog==6.12.0
confection==0.1.5
cymem==2.0.13
decorator==5.3.1
demucs==4.1.0
docopt==0.6.2
dora_search==0.1.13
einops==0.8.2
encodec==0.1.1
filelock==3.32.3
flashy==0.0.2
fsspec==2026.7.0
hf-xet==1.6.0
huggingface_hub==0.36.2
hydra-colorlog==1.2.0
hydra-core==1.3.2
idna==3.19
Jinja2==3.1.6
joblib==1.6.0
julius==0.2.8
lameenc==1.8.4
langcodes==3.5.1
lazy-loader==0.5
librosa==0.11.0
llvmlite==0.49.0
MarkupSafe==3.0.3
mpmath==1.3.0
msgpack==1.2.2
murmurhash==1.0.15
narwhals==2.26.0
networkx==3.6.1
num2words==0.5.14
numba==0.67.0
numpy==2.4.6
nvidia-cublas-cu12==12.1.3.1
nvidia-cuda-cupti-cu12==12.1.105
nvidia-cuda-nvrtc-cu12==12.1.105
nvidia-cuda-runtime-cu12==12.1.105
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.0.2.54
nvidia-curand-cu12==10.3.2.106
nvidia-cusolver-cu12==11.4.5.107


In [33]:
# 누락된 prety_middi, torchmetricx 설치후 재검증
!/content/miniconda3/envs/stemremix/bin/python -m pip install pretty_midi torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torchmetrics]


In [34]:
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 1
STDOUT >>> ''
STDERR >>> 'Traceback (most recent call last):\n  File "<string>", line 1, in <module>\n  File "/content/MusiConGen/audiocraft/audiocraft/__init__.py", line 24, in <module>\n    from . import data, modules, models\n  File "/content/MusiConGen/audiocraft/audiocraft/data/__init__.py", line 10, in <module>\n    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, btc_chords\n  File "/content/MusiConGen/audiocraft/audiocraft/data/info_audio_dataset.py", line 19, in <module>\n    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes\n  File "/content/MusiConGen/audiocraft/audiocraft/modules/conditioners.py", line 21, in <module>\n    import spacy\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/spacy/__init__.py", line 6, in <module>\n    from .errors import setup_default_warnings\n  File "/content/miniconda3/envs/stemremix/lib/python3.11/site-packages/spacy/errors.py", line 3, in

In [35]:
# numpy 버전이 호환이 안됨 --> 언제 버전 바뀌었는지 확인
!/content/miniconda3/envs/stemremix/bin/python -m pip show numpy
print("==============================================")
!/content/miniconda3/envs/stemremix/bin/python -m pip show thinc

Name: numpy
Version: 2.4.6
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License-Expression: BSD-3-Clause AND 0BSD AND MIT AND Zlib AND CC0-1.0
Location: /content/miniconda3/envs/stemremix/lib/python3.11/site-packages
Requires: 
Required-by: blis, encodec, flashy, librosa, numba, pandas, pretty_midi, pystoi, scikit-learn, scipy, soundfile, soxr, spacy, thinc, torchmetrics, transformers, xformers
Name: thinc
Version: 8.1.12
Summary: A refreshing functional take on deep learning, compatible with your favorite libraries
Home-page: https://github.com/explosion/thinc
Author: Explosion
Author-email: contact@explosion.ai
License: MIT
Location: /content/miniconda3/envs/stemremix/lib/python3.11/site-packages
Requires: blis, catalogue, confection, cymem, murmurhash, numpy, packaging, preshed, pydantic, setuptools, srsly, wasabi
Required-by: spacy


In [36]:
# 호환버전 numpy로 재설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install "numpy==1.24.4" --force-reinstall

  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 3.0.5 requires numpy>=1.26.0; python_version < "3.14", but you have numpy 1.24.4 which is incompatible.


In [37]:
# thic과 pandas 모두 호환되는 버전으로 재설치
!/content/miniconda3/envs/stemremix/bin/python -m pip install "numpy==1.26.4" --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 39.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4


In [38]:
# 재검증 완료 -> thic, pandas 모두와 호환됨
import subprocess
result = subprocess.run(
    ['/content/miniconda3/envs/stemremix/bin/python', '-c',
     'import sys; sys.path.insert(0, "/content/MusiConGen/audiocraft"); import audiocraft; print("OK:", audiocraft.__file__)'],
    capture_output=True, text=True
)
print("returncode:", result.returncode)
print("STDOUT >>>", repr(result.stdout))
print("STDERR >>>", repr(result.stderr))

returncode: 0
STDOUT >>> 'OK: /content/MusiConGen/audiocraft/audiocraft/__init__.py\n'
STDERR >>> ''


In [39]:
# stemremix환경에 ipykernel 설치/등록
# stemremix conda 환경을 Colab의 Jupyter 커널로 등록
# 실행 후: 상단 메뉴 "런타임 > 런타임 유형 변경"이 아니라,
#          우측 상단 "연결됨" 옆 화살표 또는 "런타임 > 다른 런타임에 연결" 근처의
#          커널 선택 메뉴(또는 노트북 설정의 "Kernel" 드롭다운)에서
#          "Python (stemremix)" 을 선택해야 적용됨.
# Colab 버전에 따라 커널 드롭다운 위치가 다를 수 있음 — 안 보이면
# 상단 메뉴에서 "런타임 > 런타임 유형 변경" 옆에 있는 커널 아이콘을 확인.

!/content/miniconda3/envs/stemremix/bin/python -m pip install -q ipykernel
!/content/miniconda3/envs/stemremix/bin/python -m ipykernel install --user --name stemremix --display-name "Python (stemremix)"

print("등록 완료. 이제 Colab에서 커널을 'Python (stemremix)'로 전환한 뒤,")
print("아래 확인 셀을 새 커널에서 실행해서 stemremix 환경이 맞는지 검증할 것.")

Installed kernelspec stemremix in /root/.local/share/jupyter/kernels/stemremix
등록 완료. 이제 Colab에서 커널을 'Python (stemremix)'로 전환한 뒤,
아래 확인 셀을 새 커널에서 실행해서 stemremix 환경이 맞는지 검증할 것.


In [40]:
# 새 커널에서 실행 -> audiocraft import 성공 여부 확인
# 커널을 "Python (stemremix)"로 전환한 뒤 이 셀을 실행해서 확인
# sys.executable이 /content/miniconda3/envs/stemremix/bin/python 이어야 정상

import sys
print("현재 커널 파이썬:", sys.executable)

import numpy, torch, xformers
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("xformers:", xformers.__version__)
print("CUDA available:", torch.cuda.is_available())

# audiocraft 경로를 이번엔 커널 자체에 등록 (매 셀마다 sys.path.insert 반복 안 하도록)
sys.path.insert(0, "/content/MusiConGen/audiocraft")
import audiocraft
print("audiocraft OK:", audiocraft.__file__)

현재 커널 파이썬: /usr/bin/python3


ModuleNotFoundError: No module named 'xformers'

## 2. 체크포인트 다운로드

공식 안내대로 HuggingFace(`Cyan0731/MusiConGen`)에서 `compression_state_dict.bin`, `state_dict.bin`을 받아
`audiocraft/ckpt/musicongen/`에 배치. (README 기준 확인된 절차 — 임의 추정 아님)

In [ ]:
from huggingface_hub import hf_hub_download
import os

ckpt_dir = "audiocraft/ckpt/musicongen"
os.makedirs(ckpt_dir, exist_ok=True)

for fname in ["compression_state_dict.bin", "state_dict.bin"]:
    path = hf_hub_download(repo_id="Cyan0731/MusiConGen", filename=fname, local_dir=ckpt_dir)
    print("downloaded:", path)


compression_state_dict.bin: reconstructing file:   0%|          |  0.00B /   589B            

compression_state_dict.bin: downloading bytes:           |  0.00B            

downloaded: /content/MusiConGen/audiocraft/ckpt/musicongen/compression_state_dict.bin


state_dict.bin: reconstructing file:   0%|          |  0.00B / 2.77GB            

state_dict.bin: downloading bytes:           |  0.00B            

downloaded: /content/MusiConGen/audiocraft/ckpt/musicongen/state_dict.bin


## 3. 분석 도구 설치 (madmom) + config.yaml 로드

In [ ]:
!pip install madmom pyyaml librosa soundfile -q

import yaml

with open("../stem-remix-assistant/experiments/exp_002_musicongen/config.yaml") as f:
    cfg = yaml.safe_load(f)

TARGET_STEM = cfg["target_stem"]
PROMPT = cfg["prompt"]
DURATION_SWEEP = cfg["duration_sec_sweep"]
SEED = cfg["seed"]

print("target_stem:", TARGET_STEM)
print("prompt:", PROMPT)
print("duration_sec_sweep:", DURATION_SWEEP)
print("seed:", SEED)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 54.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.3 MB/s eta 0:00:00
target_stem: drums
prompt: punchier trap-style drums, same tempo
duration_sec_sweep: [10, 15, 20]
seed: 42


## 4. 원곡·원곡 스템에서 BPM/코드/비트 추출

EXP-001에서 만든 `docs/samples/draft_0.wav`(원곡)와 `docs/samples/{target_stem}.wav`(원곡 스템)를 기준으로 madmom 분석. 여기서 얻은 BPM/코드가 MusiConGen 조건화 입력이자, 나중에 재생성 스템과 비교할 기준값.

In [ ]:
from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor
from madmom.features.tempo import TempoEstimationProcessor
from madmom.features.chords import DeepChromaChordRecognitionProcessor
from madmom.features.chroma import DeepChromaProcessor
from madmom.features.onsets import CNNOnsetProcessor, OnsetPeakPickingProcessor
import numpy as np

ORIG_TRACK = "../stem-remix-assistant/docs/samples/draft_0.wav"
ORIG_STEM = f"../stem-remix-assistant/docs/samples/{TARGET_STEM}.wav"

def get_bpm(path):
    act = RNNBeatProcessor()(path)
    tempo_proc = TempoEstimationProcessor(fps=100)
    tempi = tempo_proc(act)
    return float(tempi[0][0])  # 가장 confidence 높은 BPM 후보

def get_beats(path):
    act = RNNBeatProcessor()(path)
    beat_proc = DBNBeatTrackingProcessor(fps=100)
    return beat_proc(act)

def get_onsets(path):
    act = CNNOnsetProcessor()(path)
    onset_proc = OnsetPeakPickingProcessor(fps=100)
    return onset_proc(act)

def get_chords(path):
    chroma = DeepChromaProcessor()(path)
    chord_proc = DeepChromaChordRecognitionProcessor()
    return chord_proc(chroma)  # [(start, end, chord_label), ...]

orig_bpm = get_bpm(ORIG_TRACK)
orig_beats = get_beats(ORIG_STEM)
orig_onsets = get_onsets(ORIG_STEM)
orig_chords = get_chords(ORIG_TRACK)

print("원곡 BPM:", round(orig_bpm, 1))
print("원곡 코드 진행(앞 4개):", orig_chords[:4])


ImportError: cannot import name 'MutableSequence' from 'collections' (/usr/lib/python3.13/collections/__init__.py)

## 5. MusiConGen 조건화 입력 준비

madmom이 추출한 코드 라벨(`C:maj`, `A:min` 등)을 MusiConGen의 `generate_with_chords_and_beats` 입력 형식(공백으로 구분된 코드 시퀀스 문자열)으로 변환.

**주의**: MusiConGen 공식 스크립트(`generate_chord_beat.py`)에는 `seed` 파라미터가 없음 — 재현성 확보를 위해 아래처럼 `torch.manual_seed()`를 생성 직전에 직접 호출.

In [ ]:
def chords_to_musicongen_format(chord_events, n_bars=8):
    # (start, end, label) 리스트를 MusiConGen 예제 형식("C G A:min F")에 맞춰
    # 마디 단위로 대표 코드만 뽑아 공백 구분 문자열로 변환.
    # 실제 마디 길이(원곡 BPM/박자 기준)에 맞춰 보정 필요 — 여기서는 단순 등간격 샘플링.
    labels = [c[2] for c in chord_events if c[2] != "N"]
    if not labels:
        labels = ["C"]
    step = max(1, len(labels) // n_bars)
    sampled = labels[::step][:n_bars]
    return " ".join(sampled)

chord_str = chords_to_musicongen_format(orig_chords)
bpm_int = int(round(orig_bpm))
print("MusiConGen 조건화 코드 문자열:", chord_str)
print("MusiConGen 조건화 BPM:", bpm_int)


## 6. MusiConGen 로드 및 duration_sec 스윕 생성

In [ ]:
import torch
import audiocraft
from audiocraft.data.audio import audio_write

musicgen = audiocraft.models.MusicGen.get_pretrained("./ckpt/musicongen")

results_dir = "../stem-remix-assistant/experiments/exp_002_musicongen/outputs"
import os
os.makedirs(results_dir, exist_ok=True)

generated_paths = {}

for duration in DURATION_SWEEP:
    torch.manual_seed(SEED)  # 스윕 전 구간 동일 seed 고정

    musicgen.set_generation_params(duration=duration, extend_stride=duration // 2, top_k=250)

    wav = musicgen.generate_with_chords_and_beats(
        [PROMPT],
        [chord_str],
        [bpm_int],
        [4],  # 4/4 박자 가정
    )

    out_path = f"{results_dir}/musicongen_{TARGET_STEM}_{duration}s_seed{SEED}"
    audio_write(out_path, wav[0].cpu(), musicgen.sample_rate, strategy="loudness", loudness_compressor=True)
    generated_paths[duration] = out_path + ".wav"
    print(f"생성 완료: duration={duration}s -> {out_path}.wav")


## 7. 재생성 트랙 Demucs 재분리 → target_stem만 추출

In [ ]:
!pip install demucs -q

import subprocess

demucs_out_dir = f"{results_dir}/demucs_separated"

extracted_stems = {}
for duration, path in generated_paths.items():
    subprocess.run(
        ["python", "-m", "demucs", "-n", "htdemucs", "-o", demucs_out_dir, path],
        check=True,
    )
    track_name = os.path.splitext(os.path.basename(path))[0]
    stem_path = f"{demucs_out_dir}/htdemucs/{track_name}/{TARGET_STEM}.wav"
    extracted_stems[duration] = stem_path
    print(f"duration={duration}s -> {stem_path}")


## 8. Alignment Engine — Beat Align / Transient Align / Time Stretch

재생성 스템을 원곡 스템 기준으로 보정. 세 단계는 독립적으로 나눠서, 어느 단계에서 얼마나 개선되는지 확인 가능하게 구성.

- **Beat Align**: 재생성 스템 첫 비트를 원곡 스템 첫 비트에 맞춰 앞뒤로 자름(오프셋 보정)
- **Time Stretch**: 재생성 스템 BPM과 원곡 BPM의 비율만큼 librosa로 타임 스트레칭
- **Transient Align**: 온셋 단위 미세 보정 (여기서는 온셋 오프셋 표준편차만 측정, 실제 워핑은 다음 단계 과제로 남김)

In [ ]:
import librosa
import soundfile as sf

def beat_align(gen_path, gen_beats, orig_beats, out_path):
    y, sr = librosa.load(gen_path, sr=None)
    if len(gen_beats) == 0 or len(orig_beats) == 0:
        sf.write(out_path, y, sr)
        return out_path
    offset_sec = gen_beats[0] - orig_beats[0]
    offset_samples = int(offset_sec * sr)
    y_aligned = y[max(0, offset_samples):] if offset_samples > 0 else np.concatenate([np.zeros(-offset_samples), y])
    sf.write(out_path, y_aligned, sr)
    return out_path

def time_stretch_to_bpm(path, current_bpm, target_bpm, out_path):
    y, sr = librosa.load(path, sr=None)
    rate = current_bpm / target_bpm if target_bpm else 1.0
    y_stretched = librosa.effects.time_stretch(y, rate=rate)
    sf.write(out_path, y_stretched, sr)
    return out_path

aligned_stems = {}
alignment_dir = f"{results_dir}/aligned"
os.makedirs(alignment_dir, exist_ok=True)

for duration, stem_path in extracted_stems.items():
    gen_bpm = get_bpm(stem_path)
    gen_beats = get_beats(stem_path)

    stretched_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_stretched.wav"
    time_stretch_to_bpm(stem_path, gen_bpm, orig_bpm, stretched_path)

    gen_beats_stretched = get_beats(stretched_path)
    final_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_final.wav"
    beat_align(stretched_path, gen_beats_stretched, orig_beats, final_path)

    aligned_stems[duration] = final_path
    print(f"duration={duration}s: 원본 BPM {round(gen_bpm,1)} -> 보정 후 원곡 BPM {round(orig_bpm,1)}에 정렬")


## 9. 정량 지표 계산 (BPM 오차 / Beat alignment / Onset alignment)

`docs/PROPOSAL.md` 4.3절 평가 방법 기준. 세 duration 후보를 비교해 최적값 후보를 정함.

In [ ]:
import pandas as pd

def beat_alignment_score(beats_a, beats_b):
    # 두 비트 시퀀스를 가까운 것끼리 매칭했을 때의 평균 오차(초) — 작을수록 정렬 잘 됨
    if len(beats_a) == 0 or len(beats_b) == 0:
        return None
    errors = [min(abs(a - b) for b in beats_b) for a in beats_a]
    return float(np.mean(errors))

def onset_alignment_score(onsets_a, onsets_b):
    if len(onsets_a) == 0 or len(onsets_b) == 0:
        return None
    errors = [min(abs(a - b) for b in onsets_b) for a in onsets_a]
    return float(np.mean(errors))

rows = []
for duration, final_path in aligned_stems.items():
    final_bpm = get_bpm(final_path)
    final_beats = get_beats(final_path)
    final_onsets = get_onsets(final_path)

    rows.append({
        "duration_sec": duration,
        "bpm_error": round(abs(final_bpm - orig_bpm), 2),
        "beat_alignment_sec": round(beat_alignment_score(final_beats, orig_beats), 3)
            if beat_alignment_score(final_beats, orig_beats) is not None else None,
        "onset_alignment_sec": round(onset_alignment_score(final_onsets, orig_onsets), 3)
            if onset_alignment_score(final_onsets, orig_onsets) is not None else None,
    })

results_df = pd.DataFrame(rows).sort_values("duration_sec")
results_df.to_csv(f"{results_dir}/duration_sweep_results.csv", index=False)
results_df


## 10. 다음 작업

1. 위 표에서 `bpm_error` + `beat_alignment_sec` + `onset_alignment_sec`가 가장 낮은 `duration_sec`를 최적값으로 선택
2. `experiments/exp_002_musicongen/config.yaml`의 `duration_sec`에 확정값 기록, `duration_sec_sweep` 필드는 그대로 이력으로 남김
3. 청취 평가(사람이 직접 듣고 자연스러움 판단) 진행 — `docs/experiments/model_comparison.md`에 기록
4. `experiments/exp_002_musicongen/README.md`의 "결과"·"결론" 섹션 채우기
5. 동일한 duration_sec/seed 스윕 설계를 EXP-003(MusicGen-Melody/Style)에도 적용해 공정 비교 조건 맞추기